In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
dataset = pd.read_csv('Age_Salary_Buy.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values
print(dataset.iloc[:10, :])
X[:,1] = X[:,1]/1000
print(X[:10,:])

   Age  EstimatedSalary  Purchased
0   19            19000          0
1   35            20000          0
2   26            43000          0
3   27            57000          0
4   19            76000          0
5   27            58000          0
6   27            84000          0
7   32           150000          1
8   25            33000          0
9   35            65000          0
[[ 19  19]
 [ 35  20]
 [ 26  43]
 [ 27  57]
 [ 19  76]
 [ 27  58]
 [ 27  84]
 [ 32 150]
 [ 25  33]
 [ 35  65]]


In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

In [4]:
# Keep copies for plotting
X_train_orig = X_train.copy()
X_test_orig  = X_test.copy()

In [5]:
print(X_train_orig[:10,:])

[[ 44  39]
 [ 32 120]
 [ 38  50]
 [ 32 135]
 [ 52  21]
 [ 53 104]
 [ 39  42]
 [ 38  61]
 [ 36  50]
 [ 36  63]]


In [6]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

# Compute the means and standard deviations of features w.r.t
# training set. Transform X_train
X_train = sc.fit_transform(X_train)

# Using the scaler computed above, transform X_test
X_test = sc.transform(X_test)

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

In [9]:
# model: 2 -> 4 -> 1, ReLU + Sigmoid
model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)


X = torch.from_numpy(X_train).float()
y = torch.from_numpy(y_train).float()

loss_fn = nn.BCELoss()
opt = optim.SGD(model.parameters(), lr=0.05, momentum=0.9)

batch_size = 32
N = X.size(0)

for epoch in range(500):
    perm = torch.randperm(N)          # shuffle (like np.random.permutation)
    Xs, ys = X[perm], y[perm]

    for i in range(0, N, batch_size):
        xb = Xs[i:i+batch_size]
        yb = ys[i:i+batch_size]

        y_pred = model(xb)
        yb = yb.unsqueeze(1)
        loss = loss_fn(y_pred, yb)

        opt.zero_grad()
        loss.backward()
        opt.step()

In [10]:
def predict_proba(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).squeeze().numpy()

def predict(model, X, threshold=0.5):
    probs = predict_proba(model, X)
    return (probs >= threshold).astype(int)

In [13]:
y_pred = predict(model, X)

In [34]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_train, y_pred)
print(cm)
print(accuracy_score(y_train, y_pred))

[[175  14]
 [ 10 101]]
0.92


In [15]:
y_pred_test = predict(model, torch.from_numpy(X_test).float())

In [17]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred_test)
print(cm)
print(accuracy_score(y_test, y_pred_test))

[[64  4]
 [ 3 29]]
0.93
